In [1]:
import pandas as pd
import numpy as np 

In [6]:
def load_data():
    train = pd.read_csv("../data/raw/train.csv")
    test = pd.read_csv("../data/raw/test.csv")
    return train,test

In [91]:
pd.set_option("display.max_columns",None)
train,test = load_data()

## Column Descriptions

| Column | Type | Description |
|---|---|---|
| `id` | ID | Row identifier. Not a feature; drop before training. |
| `amount_usd` | Numeric | Transaction amount in US dollars. |
| `hours_since_last_txn` | Numeric | Hours since the card's previous transaction. Very small values suggest rapid repeated use. |
| `txn_count_last_24h` | Numeric | Number of transactions on the card in the last 24 hours. |
| `distance_from_home_km` | Numeric | Distance between the transaction location and the customer's home, in km. |
| `card_age_months` | Numeric | How long the card has existed, in months. |
| `customer_age` | Numeric | Age of the cardholder in years. |
| `account_balance_usd` | Numeric | Account balance in US dollars at the time of the transaction. |
| `cvv_retry_count` | Numeric | Number of wrong CVV (security code) entries before this transaction. |
| `velocity_score` | Numeric | Summary score of how fast and how much the card is being used. Higher means more unusual activity. |
| `time_of_day_hour` | Numeric | Hour of the transaction (0–23). EDA showed no meaningful link to fraud. |
| `day_of_week` | Categorical | Day of the week of the transaction. |
| `merchant_risk_score` | Numeric | Risk score assigned to the merchant. Higher means riskier. |
| `prior_disputes` | Numeric | Number of past transactions the customer disputed. |
| `merchant_category` | Categorical | Type of business (e.g. grocery, electronics, travel). |
| `card_type` | Categorical | Type of card (e.g. credit, debit, prepaid). |
| `auth_method` | Categorical | How the transaction was verified (e.g. PIN, OTP, 3D Secure). |
| `channel` | Categorical | Where the transaction happened (POS, Online, In-App). |
| `device_type` | Categorical | Device used (e.g. iPhone, Android Phone, Tablet, Windows PC). |
| `is_foreign_transaction` | Boolean | True if the transaction is in a different country from the card's home country. |
| `is_new_merchant` | Boolean | True if the customer has never used this merchant before. |
| `used_vpn` | Boolean | True if the connection went through a VPN, which can hide the real location. |
| `ip_country_mismatch` | Boolean | True if the IP address country differs from the card's or customer's country. |
| `billing_shipping_mismatch` | Boolean | True if the billing and shipping addresses differ. |
| `is_ai_generated_scam_attempt` | Boolean | True if the transaction was flagged as part of an AI-generated scam (e.g. deepfake call, phishing). Check how strongly it predicts fraud. |
| `is_fraud` | Target | 1 if the transaction is fraud, 0 if legitimate. This is what the model predicts. |

In [103]:
def add_features(df):
    df = df.copy()
    df["amount_to_balance"] = df["amount_usd"] / (df["account_balance_usd"] + 1)
    df["risk_exposure"] = df["merchant_risk_score"] * df["amount_usd"]
    df["risk_count"] = df[["is_foreign_transaction", "is_new_merchant", "used_vpn",
                           "ip_country_mismatch", "billing_shipping_mismatch",
                           "is_ai_generated_scam_attempt"]].sum(axis=1)
    df["cvv_scam"] = df["cvv_retry_count"] * df["is_ai_generated_scam_attempt"]
    return df
train = add_features(train)
test = add_features(test)

In [115]:
def x_y(train):
    x = train.drop(columns=["id","is_fraud"])
    y = train["is_fraud"]
    return x,y
x,y = x_y(train)

In [129]:
from sklearn.model_selection import train_test_split
def train_test_split(x,y):
    x_train,x_val,y_train,y_val = train_test_split(
        x,y, test_size= 0.2,stratify=y,random_state = 42)
    return x_train,x_val,y_train,y_val

In [130]:
x_train,x_val,y_train,y_val = train_test_split(x,y)

TypeError: train_test_split() got an unexpected keyword argument 'test_size'